# 🦕 search-o-SAURS — Boolean Search Pipeline

This notebook runs the **complete IR pipeline** using the project's Python modules.
No code is duplicated — everything is imported from the source files.

**Pipeline:** `data/cran.all.1400` → Preprocess → Index → Query


In [ ]:
import importlib.util, os

def load_module(name, filepath):
    """Import a module from a hyphenated filename."""
    spec = importlib.util.spec_from_file_location(name, filepath)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

preprocessor = load_module("preprocessor", "search-o-SAURS_preprocess.py")
indexer      = load_module("indexer",      "search-o-SAURS_indexer.py")
searcher     = load_module("searcher",     "search-o-SAURS_search.py")

print("✓ All modules loaded.")


## Step 1: Preprocess the Corpus
Runs: **Tokenize → Normalize → Stop words → Stem → Deduplicate**


In [ ]:
PROCESSED_FILE = os.path.join("output", "search-o-SAURS_processed.all")

if os.path.exists(PROCESSED_FILE):
    size = os.path.getsize(PROCESSED_FILE)
    print(f"✓ Preprocessed file already exists ({size:,} bytes). Skipping.")
    print(f"  Delete '{PROCESSED_FILE}' and re-run this cell to regenerate.")
else:
    preprocessor.main()


## Step 2: Build the Inverted Index
Creates a **sorted inverted index** with document frequencies.
Format: `token df docid1,docid2,...`


In [ ]:
INDEX_FILE = os.path.join("output", "search-o-SAURS_cran.index")

if os.path.exists(INDEX_FILE):
    size = os.path.getsize(INDEX_FILE)
    vocab, maxid = searcher.read_index_header(INDEX_FILE)
    print(f"✓ Index already exists ({size:,} bytes). Skipping.")
    print(f"  {vocab:,} terms, max docid = {maxid}")
    print(f"  Delete '{INDEX_FILE}' and re-run this cell to regenerate.")
else:
    indexer.main()


## Step 3: Boolean Search 🔍

**Edit the `QUERY` variable below and run the cell.**

Supports: `term1 AND term2` or `term1 OR term2`

British spellings are automatically normalized (e.g., `behaviour` → `behavior`).


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║   EDIT YOUR QUERY HERE                               ║
# ╚══════════════════════════════════════════════════════╝
QUERY = "aerodynamic AND experimental"

# ────────────────────────────────────────────────────────
OUTPUT_FILE = os.path.join("output", "results", "search-o-SAURS_results.txt")
os.makedirs(os.path.join("output", "results"), exist_ok=True)

# Parse query
parts = QUERY.split()
if len(parts) != 3 or parts[1].upper() not in ("AND", "OR"):
    print(f"Error: Query must be 'term1 AND term2' or 'term1 OR term2'. Got: '{QUERY}'")
else:
    term1_raw, operator, term2_raw = parts[0], parts[1].upper(), parts[2]

    # Preprocess query terms (same pipeline as documents)
    term1 = searcher.preprocess_query_term(term1_raw)
    term2 = searcher.preprocess_query_term(term2_raw)

    print(f"┌─ Query: '{QUERY}'")
    print(f"│  Processed: '{term1}' {operator} '{term2}'")

    # Binary search for each term on the index file
    L1, df1 = searcher.binary_search_index(INDEX_FILE, term1)
    L2, df2 = searcher.binary_search_index(INDEX_FILE, term2)

    if L1 is None: L1 = []
    if L2 is None: L2 = []

    print(f"│  Postings for '{term1}': {len(L1)} documents")
    print(f"│  Postings for '{term2}': {len(L2)} documents")

    # Merge
    if operator == "AND":
        results = searcher.intersect_postings(L1, L2)
    else:
        results = searcher.union_postings(L1, L2)

    print(f"│  Result ({operator}): {len(results)} documents")
    print(f"└─ Results written to: {OUTPUT_FILE}")

    # Write results
    with open(OUTPUT_FILE, "w") as f:
        for docid in results:
            f.write(f"{docid}\n")

    # Display
    print(f"\nMatching Document IDs ({len(results)}):")
    for i in range(0, len(results), 15):
        print("  " + ", ".join(str(d) for d in results[i:i+15]))
